<a href="https://colab.research.google.com/github/atomicSteiner/HealthcareSBERT/blob/main/InformationRetrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import wilcoxon
from tqdm import tqdm

In [ ]:
#Information retrival setting up

N_QUERIES = 500        # Queries number
TOP_K = [1, 5, 10, 50, 1000]     # Metrics cutoff
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

baseline_emb = np.load("/content/drive/MyDrive/NLP/embeddings_baseline.npy")
finetuned_emb = np.load("/content/drive/MyDrive/NLP/fine_tuned_embeddings.npy")

df = pd.read_csv("/content/drive/MyDrive/NLP/ohsumed_cleaned_test.csv")
texts = df['abstract'].tolist()
mesh_labels = df['mesh_terms'].tolist()

print(mesh_labels[0])

Acetaldehyde/*ME; Buffers; Catalysis; HEPES/PD; Nuclear Magnetic Resonance; Phosphates/*PD; Protein Binding; Ribonuclease, Pancreatic/AI/*ME; Support, U.S. Gov't, Non-P.H.S.; Support, U.S. Gov't, P.H.S..


In [ ]:
#Mesh term preprocessing

def preprocess_mesh_terms(labels):
  preprocessed_mesh = set()
  for lbl in labels:
    items = set(sentence for sentence in lbl.split(";"))
    preprocessed_mesh.update(items)
  return list(preprocessed_mesh)

mesh_labels = preprocess_mesh_terms(mesh_labels)

In [ ]:
# =========================
# 2. SAMPLING DELLE QUERY
# =========================
query_indices = np.random.choice(baseline_emb.shape[0], size=N_QUERIES, replace=False)
queries_baseline = baseline_emb[query_indices]
queries_finetuned = finetuned_emb[query_indices]
query_labels = [mesh_labels[i] for i in query_indices]

In [ ]:
# =========================
# 3. GROUND TRUTH (MeSH overlap)
# =========================
def get_relevance_set(query_label, all_labels):
  ground_truth = set(np.where([len(set(query_label) & set(lbl)) > 0 for lbl in all_labels])[0])
  return ground_truth

relevance_sets = [get_relevance_set(lbl, mesh_labels) for lbl in query_labels]

avg_gt_size = np.mean([len(r) for r in relevance_sets])
print("Average ground truth size per query:", avg_gt_size)


Average ground truth size per query: 313430.878


In [ ]:
# =========================
# 4. METRICHE IR
# =========================

def recall_at_k(ranked, relevant, k):
  return len(set(ranked[:k]) & relevant) / len(relevant) if len(relevant) > 0 else 0

def average_precision(ranked, relevant):
    hits, sum_prec = 0, 0
    for i, doc in enumerate(ranked):
        if doc in relevant:
            hits += 1
            sum_prec += hits / (i + 1)
    return sum_prec / len(relevant) if len(relevant) > 0 else 0

def reciprocal_rank(ranked, relevant):
    for i, doc in enumerate(ranked):
        if doc in relevant:
            return 1 / (i + 1)
    return 0

def ndcg_at_k(ranked, relevant, k):
    dcg = sum([1 / np.log2(i + 2) for i, doc in enumerate(ranked[:k]) if doc in relevant])
    ideal_dcg = sum([1 / np.log2(i + 2) for i in range(min(len(relevant), k))])
    return dcg / ideal_dcg if ideal_dcg > 0 else 0

In [ ]:
# =========================
# 5. EVALUATION FUNCTION
# =========================
def evaluate_model(query_embs, corpus_embs, relevance_sets):
    sims = cosine_similarity(query_embs, corpus_embs)
    metrics_per_query = {
        'MRR': [], 'MAP': [],
        **{f'R@{k}': [] for k in TOP_K},
        **{f'nDCG@{k}': [] for k in TOP_K}
    }

    for i in tqdm(range(len(query_embs)), desc="Evaluating retrieval"):
        ranked = np.argsort(-sims[i])  # ordina per similarità decrescente
        relevant = relevance_sets[i]

        metrics_per_query['MRR'].append(reciprocal_rank(ranked, relevant))
        metrics_per_query['MAP'].append(average_precision(ranked, relevant))

        for k in TOP_K:
            metrics_per_query[f'R@{k}'].append(recall_at_k(ranked, relevant, k))
            metrics_per_query[f'nDCG@{k}'].append(ndcg_at_k(ranked, relevant, k))

    avg_metrics = {k: np.mean(v) for k, v in metrics_per_query.items()}
    return avg_metrics, metrics_per_query


In [ ]:
# =========================
# 6. PARALLEL EVALUATION
# =========================
baseline_avg, baseline_perq = evaluate_model(queries_baseline, baseline_emb, relevance_sets)
finetuned_avg, finetuned_perq = evaluate_model(queries_finetuned, finetuned_emb, relevance_sets)

Evaluating retrieval: 100%|██████████| 500/500 [00:39<00:00, 12.73it/s]


In [ ]:
# =========================
# 7. RISULTATI
# =========================
print("\n=== Baseline IR metrics ===")
for k, v in baseline_avg.items():
    print(f"{k}: {v:.4f}")

print("\n=== Fine-tuned IR metrics ===")
for k, v in finetuned_avg.items():
    print(f"{k}: {v:.4f}")


=== Baseline IR metrics ===
MRR: 1.0000
MAP: 0.6269
R@1: 0.0000
R@5: 0.0000
R@10: 0.0000
R@50: 0.0002
R@1000: 0.0032
nDCG@1: 1.0000
nDCG@5: 1.0000
nDCG@10: 0.9999
nDCG@50: 0.9999
nDCG@1000: 1.0000

=== Fine-tuned IR metrics ===
MRR: 1.0000
MAP: 0.6269
R@1: 0.0000
R@5: 0.0000
R@10: 0.0000
R@50: 0.0002
R@1000: 0.0032
nDCG@1: 1.0000
nDCG@5: 1.0000
nDCG@10: 1.0000
nDCG@50: 1.0000
nDCG@1000: 0.9999
